<a href="https://colab.research.google.com/github/praveena-muvva/hf-llm/blob/main/06_finetuning_end_to_end.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers datasets peft accelerate bitsandbytes trl

import torch

In [ ]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_properties(0).total_memory)

In [ ]:
#*****************
#QLoRA compresses the base model weights from 16-bit to 4-bit which saves memory during loading and training
#The LoRA adapters (The diff matrices we train) stay in higher precision
#A 7B parameter usually needs about 28GB memory, but QLoRA helps in loading this here within 15GB available memory

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training
import torch

# 1. Configure QLoRA
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True, #Extra compression
)

#2. Load Mistral 7B
print("Loading Mistral-7B ......")
model_name = "mistralai/Mistral-7B-v0.1"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config = bnb_config,
    device_map = "auto", #Auto map to available GPU
    trust_remote_code = True,
)

In [ ]:
#3. Prepare for LoRA training
model = prepare_model_for_kbit_training(model)

print(f"Model loaded and the memory used: {model.get_memory_footprint() / 1e9:.2f} GB")

#4. Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token #Mistral doesn't have padding token, hence using eos
tokenizer.padding_side = "right"

print(f"Tokenizer ready! Vocab size: {len(tokenizer):,}")

In [ ]:
#***********
#As LoRA works on adapters (The diff matrices), between attention layer vs feedforward layer, which one does LoRA apply to?
#Attention layer as that's the main part where relation between each word is saved and understanding comes from and optimizing it gives more value
#Feedforward layers are more like memory storage

In [ ]:
from peft import LoraConfig, get_peft_model

#Configure LoRA
lora_config = LoraConfig(
    r = 16, #Rank: size of adapter matrix
    lora_alpha = 32, #scaling factor usually 2x rank
    target_modules=[#Which layer to add LoRA to
        "q_proj", #Quert projection
                       "k_proj", #Key projection
                       "v_proj", #value projection
                       "o_proj", #Output projection
    ],
    lora_dropout=0.05, #Output from regularization
    bias="none", #Don't adapt bias paramters
    task_type="CAUSAL_LM" #Language modeling task
)

#Apply LoRA to the model
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()